In [3]:
import re
import ast
import pandas as pd

filename = '/home/mudryi/phd_projects/textfooler_ukr/adv_results_reviews_xml_roberta/adversaries.txt'

with open(filename, 'r', encoding='utf-8') as f:
    content = f.read().strip()

# Split entries at lines starting with "text <number>"
entries = re.split(r'\n(?=text \d+)', content)
records = []

for entry in entries:
    lines = entry.strip().splitlines()
    # Parse original sentence and label
    orig_match = re.match(r'orig sent \((\d+)\):\s*(.*)', lines[1])
    adv_match  = re.match(r'adv sent \((\d+)\):\s*(.*)', lines[2])
    repl_match = re.match(r'Replacements\s+(.+)', lines[3])

    if orig_match and adv_match and repl_match:
        orig_label = int(orig_match.group(1))
        orig_text  = orig_match.group(2)
        adv_label  = int(adv_match.group(1))
        adv_text   = adv_match.group(2)
        replacements = ast.literal_eval(repl_match.group(1))

        records.append({
            "original_text": orig_text,
            "adversarial_text": adv_text,
            "original_label": orig_label,
            "adversarial_label": adv_label,
            "all_replacements": replacements
        })

df = pd.DataFrame(records)
df.head()


,original_text,adversarial_text,original_label,adversarial_label,all_replacements
0,"Придбав 2 кабелі. На коробці напис ""Быстрая си...","Придбав 2 кабелі. На коробці напис ""Быстрая си...",3,2,"[(достатньо, задосить, 124)]"
1,"Вчора замовила, сьогодні забрала у відділенні ...","Учора замовила, сьогодні забрала у відділенні ...",4,0,"[(нарікань, ієреміада, 39), (нема, чортмає, 41..."
2,"не поганий шампунь, гарно захищає фарбоване во...","не поганий шампунь, ладно захищає фарбоване ко...",4,3,"[(фарби, рум'янець, 19), (волосся, косм, 22), ..."
3,Після бриття саме те! Зупиняє кров з ранок при...,Після бриття саме те! Тамує кровиця з передден...,4,2,"[(закупорює, закорковувати, 24), (Зупиняє, там..."
4,#моєрозпакування Зі своїми функціями справляєт...,#моєрозпакування Зі своїми функціями справляєт...,4,3,"[(заряджає, подразнити, 12), (приємна, поманли..."


In [16]:
import re
import json
from tqdm import tqdm

def extract_json_from_content(content: str) -> dict:
    """
    Extracts and parses the first JSON object found in a GPT response string,
    stripping out markdown code fences if present.
    
    Args:
        content: The raw string response from the model.
        
    Returns:
        The parsed JSON object as a dict.
    
    Raises:
        ValueError: If no JSON object can be found or parsed.
    """
    # 1. Strip markdown code fences ```json ... ```
    fence_pattern = r'```(?:json)?\s*([\s\S]*?)\s*```'
    fence_match = re.search(fence_pattern, content)
    if fence_match:
        content = fence_match.group(1)
    
    # 2. Find the JSON substring {...}
    json_pattern = r'\{[\s\S]*\}'
    json_match = re.search(json_pattern, content)
    if not json_match:
        raise ValueError(f"No JSON object found in response:\n{content}")
    
    json_str = json_match.group()
    
    # 3. Parse and return
    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to decode JSON: {e}\nJSON was:\n{json_str}")

In [22]:
df.shape

(3153, 5)

In [ ]:
import openai
import json
import os
from dotenv import load_dotenv

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

sample_df = df.sample(100, random_state=42).reset_index(drop=True)

few_shot_examples = """
Example 1:
Text: "Прекрасний товар, дуже задоволений!"
Score: 4

Example 2:
Text: "Трохи розчарований якістю."
Score: 2
"""

# 4. System prompt
system_prompt = (
    "You are a helpful assistant that rates Ukrainian product reviews on a scale from 0 to 4, "
    "where 4 means \"very good\" and 0 means \"very bad\". "
    "Given a single review text, return only a JSON object with keys:\n"
    "  - \"predicted_label\": an integer 0–4\n"
    "Do not include any additional commentary."
)

# 5. Loop over both original and adversarial texts
results = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    for variant in ["original", "adversarial"]:
        text = row[f"{variant}_text"]
        true_label = row[f"{variant}_label"]
        
        # Build the messages
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": few_shot_examples + f"\nText: \"{text}\"\nScore:"}
        ]
        
        # Call GPT-4o
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0
        )
        content = response.choices[0].message.content.strip()
        
        # Parse the JSON response
        pred = extract_json_from_content(content)
        predicted_label = pred["predicted_label"]
        
        results.append({
            "variant": variant,
            "text": text,
            "true_label": true_label,
            "predicted_label": predicted_label
        })

# 6. Turn into a DataFrame and compute accuracy
results_df = pd.DataFrame(results)
accuracy = (results_df["true_label"] == results_df["predicted_label"]).mean()
print("Overall accuracy:", accuracy)


100%|██████████| 100/100 [02:24<00:00,  1.44s/it]

Overall accuracy: 0.535


In [24]:
results_df.groupby('variant').apply(lambda x: sum(x.true_label == x.predicted_label)/len(x))

/tmp/ipykernel_571897/2894266938.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_df.groupby('variant').apply(lambda x: sum(x.true_label == x.predicted_label)/len(x))


variant
adversarial    0.47
original       0.60
dtype: float64

In [26]:
results_df.to_pickle('gpt_100_reviews_textfooelr_xml_roberta.pickle')

In [28]:
import re
import ast
import pandas as pd

filename = '/home/mudryi/phd_projects/textfooler_ukr/adv_results_news_xlm-roberta-base/adversaries.txt'

with open(filename, 'r', encoding='utf-8') as f:
    content = f.read().strip()

# Split entries at lines starting with "text <number>"
entries = re.split(r'\n(?=text \d+)', content)
records = []

for entry in entries:
    lines = entry.strip().splitlines()
    # Parse original sentence and label
    orig_match = re.match(r'orig sent \((\d+)\):\s*(.*)', lines[1])
    adv_match  = re.match(r'adv sent \((\d+)\):\s*(.*)', lines[2])
    repl_match = re.match(r'Replacements\s+(.+)', lines[3])

    if orig_match and adv_match and repl_match:
        orig_label = int(orig_match.group(1))
        orig_text  = orig_match.group(2)
        adv_label  = int(adv_match.group(1))
        adv_text   = adv_match.group(2)
        replacements = ast.literal_eval(repl_match.group(1))

        records.append({
            "original_text": orig_text,
            "adversarial_text": adv_text,
            "original_label": orig_label,
            "adversarial_label": adv_label,
            "all_replacements": replacements
        })

df = pd.DataFrame(records)
df.head()


,original_text,adversarial_text,original_label,adversarial_label,all_replacements
0,Інтерпол оголосив у розшук ексголову Renault К...,Інтерпол оголосив у розшук ексголову Renault К...,0,1,"[(втік, утекти, 19)]"
1,Airbus показала своє літаюче таксі. До 2025 ро...,Airbus показала своє літаюче таксомоторові. До...,1,4,"[(таксі, таксомотор, 9), (світу, макрокосм, 26)]"
2,Доведемо до кінця. Зеленський підтримав земель...,Доведемо до кінця. Зеленський піддакнув земель...,0,2,"[(підтримав, піддакнути, 9)]"
3,"Білоруська прокуратура почала ""розслідування"" ...","Білоруська прокуратура почала ""розслідування"" ...",2,1,"[(опозиціонера, фрондер, 19)]"
4,Рада зробила крок до електронного резидентства,Рада зробила хід до електронного резидентства,0,2,"[(крок, хід, 4)]"


In [30]:
label_list = ['бізнес', 'новини', 'політика', 'спорт', 'технології']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
df['original_label'] = df['original_label'].map(id2label)
df['adversarial_label'] = df['adversarial_label'].map(id2label)

df.head()

,original_text,adversarial_text,original_label,adversarial_label,all_replacements
0,Інтерпол оголосив у розшук ексголову Renault К...,Інтерпол оголосив у розшук ексголову Renault К...,бізнес,новини,"[(втік, утекти, 19)]"
1,Airbus показала своє літаюче таксі. До 2025 ро...,Airbus показала своє літаюче таксомоторові. До...,новини,технології,"[(таксі, таксомотор, 9), (світу, макрокосм, 26)]"
2,Доведемо до кінця. Зеленський підтримав земель...,Доведемо до кінця. Зеленський піддакнув земель...,бізнес,політика,"[(підтримав, піддакнути, 9)]"
3,"Білоруська прокуратура почала ""розслідування"" ...","Білоруська прокуратура почала ""розслідування"" ...",політика,новини,"[(опозиціонера, фрондер, 19)]"
4,Рада зробила крок до електронного резидентства,Рада зробила хід до електронного резидентства,бізнес,політика,"[(крок, хід, 4)]"


In [31]:
df.shape

(1394, 5)

In [ ]:
import openai
import json
import os
from dotenv import load_dotenv

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

sample_df = df.sample(100, random_state=42).reset_index(drop=True)

few_shot_examples = """
Example 1:
Title: "Український стартап залучив $10 млн інвестицій"
Category: бізнес

Example 2:
Title: "Кабмін затвердив нові карантинні обмеження"
Category: новини

Example 3:
Title: "Перемога збірної України над Іспанією у фіналі Євро"
Category: спорт
"""

# 2. System prompt
system_prompt = (
    "You are a helpful assistant that classifies Ukrainian news article titles into one of the following categories: "
    "['бізнес', 'новини', 'політика', 'спорт', 'технології']. "
    "Given a single title, return ONLY a JSON object with one key:\n"
    "  - \"predicted_category\": one of ['бізнес', 'новини', 'політика', 'спорт', 'технології']\n"
    "Do not include any additional commentary, markdown, or code fences."
)

# 5. Loop over both original and adversarial texts
results = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    for variant in ["original", "adversarial"]:
        text = row[f"{variant}_text"]
        true_label = row[f"{variant}_label"]
        
        # Build the messages
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": few_shot_examples + f"\nText: \"{text}\"\nScore:"}
        ]
        
        # Call GPT-4o
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0
        )
        content = response.choices[0].message.content.strip()
        
        # Parse the JSON response
        pred = extract_json_from_content(content)
        predicted_label = pred["predicted_category"]
        
        results.append({
            "variant": variant,
            "text": text,
            "true_label": true_label,
            "predicted_label": predicted_label
        })

# 6. Turn into a DataFrame and compute accuracy
results_df = pd.DataFrame(results)
accuracy = (results_df["true_label"] == results_df["predicted_label"]).mean()
print("Overall accuracy:", accuracy)


100%|██████████| 100/100 [02:00<00:00,  1.20s/it]

Overall accuracy: 0.445


In [33]:
results_df.groupby('variant').apply(lambda x: sum(x.true_label == x.predicted_label)/len(x))

/tmp/ipykernel_571897/2894266938.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_df.groupby('variant').apply(lambda x: sum(x.true_label == x.predicted_label)/len(x))


variant
adversarial    0.25
original       0.64
dtype: float64

In [34]:
results_df.to_pickle('gpt_100_news_textfooelr_xml_roberta.pickle')